In [135]:
# imports
import pandas as pd
import matplotlib.pyplot as plt

from finvizfinance.quote import finvizfinance

import os

In [136]:
# directory path
dir_path = r"..\14_stocks_analysis\00_data"
if not os.path.isdir(dir_path):
    raise FileNotFoundError(f'Directory does not exist: {dir_path}')
print(f'Using data directory: {dir_path}')

Using data directory: ..\14_stocks_analysis\00_data


In [137]:
# concatenate all tabular files into a single dataframe
files = [
    file for file in os.listdir(dir_path)
    if file.lower().endswith(('.csv', '.xlsx', '.xls'))
]
if not files:
    raise ValueError(f'No CSV/XLSX/XLS files found in: {dir_path}')

def load_table(file_name):
    file_path = os.path.join(dir_path, file_name)
    if file_name.lower().endswith('.csv'):
        return pd.read_csv(file_path)
    return pd.read_excel(file_path)

# file name have format like 2603, 2602..first two digits are year and last two are month
# when concatenating, we need to add a column with the date extracted from the file name
def extract_date(file_name):
    year = int(file_name[:2]) + 2000
    month = int(file_name[2:4])
    return pd.Timestamp(year=year, month=month, day=1)

df = pd.concat([load_table(file).assign(date=extract_date(file)) for file in sorted(files)], ignore_index=True)

# lowercase column names
df.columns = df.columns.str.lower()

# replace " " with "_" in column names
df.columns = df.columns.str.replace(" ", "_")

df.head()

,unnamed:_0,symbol,stock,market_cap,price,fair_value_(%),z-score,f-score,m-score,value_generation,date
0,1.0,Lululemon Athletica Inc.,Lululemon Athletica Inc.,20.11B,168.18,116.31,7.43,8.0,-2.78,Robust,2025-11-01
1,NaN,LULU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
2,2.0,"PayPal Holdings, Inc.","PayPal Holdings, Inc.",58.63B,60.57,100.92,1.96,7.0,-2.99,Robust,2025-11-01
3,NaN,PYPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
4,3.0,Adobe Inc.,Adobe Inc.,139.08B,324.19,82.13,8.92,7.0,-2.49,Robust,2025-11-01


In [138]:
# create a copy of symbol column named 'symbol_copy'
df['symbol_copy'] = df['symbol']

In [139]:
# the series in df.symbol_copy, should be shifted backwards by 1 row, to align with the stock name in the same row
df['symbol_copy'] = df['symbol_copy'].shift(-1)

In [140]:
# drop nan in df.unnamed:_0
df = df.dropna(subset=['unnamed:_0'])

In [141]:
# drop columns 'unnamed: 0', 'symbol'
df = df.drop(columns=['unnamed:_0', 'symbol'])

# rename column 'symbol_copy' to 'symbol'
df = df.rename(columns={'symbol_copy': 'symbol'})

# columns order 'date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation'
df = df[['date', 'symbol', 'stock', 'market_cap', 'price', 'fair_value_(%)', 'z-score', 'f-score', 'm-score', 'value_generation']]

In [142]:
df.columns

Index(['date', 'symbol', 'stock', 'market_cap', 'price', 'fair_value_(%)',
       'z-score', 'f-score', 'm-score', 'value_generation'],
      dtype='object')

In [143]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 498
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              250 non-null    datetime64[us]
 1   symbol            250 non-null    object        
 2   stock             250 non-null    object        
 3   market_cap        250 non-null    object        
 4   price             250 non-null    float64       
 5   fair_value_(%)    250 non-null    float64       
 6   z-score           250 non-null    float64       
 7   f-score           250 non-null    object        
 8   m-score           250 non-null    float64       
 9   value_generation  250 non-null    object        
dtypes: datetime64[us](1), float64(4), object(5)
memory usage: 21.5+ KB


In [144]:
# create a copy of the dataframe
df_copy = df.copy()

# keep only date, symbol columns
df_copy = df_copy[['date', 'symbol']]

# group by symbol and count the number of occurrences of each symbol
symbol_counts = df_copy.groupby('symbol').size().reset_index(name='count')

# sort the dataframe by count in descending order
symbol_counts = symbol_counts.sort_values(by='count', ascending=False)

# display the top 10 symbols with the most occurrences
print(symbol_counts.head(10))

   symbol  count
2    ADBE      5
18    CLX      5
41   FTNT      5
93   PYPL      5
20    CMG      4
7     AVY      4
17     CL      4
35     EW      4
50    HUM      4
25   DECK      4


In [145]:
# drop duplicates in df.symbol, keep last occurrence
df = df.drop_duplicates(subset='symbol', keep='last')

# merge df with symbol_counts on symbol, keeping only rows that are in df
df = df.merge(symbol_counts, on='symbol', how='inner')

# sort df by count in descending order
df = df.sort_values(by='count', ascending=False)

df.head(10)

,date,symbol,stock,market_cap,price,fair_value_(%),z-score,f-score,m-score,value_generation,count
83,2026-03-01,PYPL,"PayPal Holdings, Inc.",42.66B,444859.00,87.65,1.99,8.0,-2.54,Robust,5
72,2026-03-01,CLX,The Clorox Company,12.44B,102.28,87.39,2.44,6.0,-3.16,Robust,5
69,2026-03-01,ADBE,Adobe Inc.,98.74B,2408252.00,101.55,7.53,7.0,-3.41,Robust,5
117,2026-03-01,FTNT,"Fortinet, Inc.",59.24B,79725.00,17.65,5.22,7.0,-2.23,Robust,5
39,2026-02-01,CMG,"Chipotle Mexican Grill, Inc.",48.27B,36.71,50.78,7.74,8.0,-1.70,Robust,4
42,2026-02-01,AVY,Avery Dennison Corporation,14.49B,187.20,29.80,3.93,9.0,-2.89,Robust,4
41,2026-02-01,DECK,Deckers Outdoor Corporation,15.98B,108.74,33.70,14.21,9.0,-2.89,Robust,4
55,2026-02-01,TMO,Thermo Fisher Scientific Inc.,192.77B,512.69,12.11,4.07,8.0,-2.65,Robust,4
66,2026-02-01,JKHY,"Jack Henry & Associates, Inc.",12.20B,168.43,23.66,10.78,8.0,-2.31,Resilient,4
74,2026-03-01,LDOS,"Leidos Holdings, Inc.",20.35B,153.66,78.00,4.15,8.0,-2.51,Robust,4


In [146]:
# market cap column is object. We need to convert it to numeric, removing any non-numeric characters ("T" indicating trillions, 
# "B", indicating billions, "M" indicating millions)
# These characters are at the end of the string, so we can use str.replace to remove them and then convert to numeric
# They will be added in a new column named 'market cap um'

# take last character of market cap column and add a new column 'market_cap_um' with the value of the last character
df['market_cap_um'] = df['market_cap'].str[-1]

# replace 'T', 'B', 'M' with "" in market cap column and convert to numeric
df['market_cap'] = df['market_cap'].str.replace(r'[TBM]', '', regex=True).astype(float)


In [147]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 119 entries, 83 to 118
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              119 non-null    datetime64[us]
 1   symbol            119 non-null    object        
 2   stock             119 non-null    object        
 3   market_cap        119 non-null    float64       
 4   price             119 non-null    float64       
 5   fair_value_(%)    119 non-null    float64       
 6   z-score           119 non-null    float64       
 7   f-score           119 non-null    object        
 8   m-score           119 non-null    float64       
 9   value_generation  119 non-null    object        
 10  count             119 non-null    int64         
 11  market_cap_um     119 non-null    object        
dtypes: datetime64[us](1), float64(5), int64(1), object(5)
memory usage: 12.1+ KB


In [148]:
# if market cap um is M, market cap / 1000, if market cap um is T, market cap * 1000, if market cap um is B, market cap / 1
def convert_market_cap(row):
    if row['market_cap_um'] == 'M':
        return row['market_cap'] / 1000
    elif row['market_cap_um'] == 'T':
        return row['market_cap'] * 1000
    elif row['market_cap_um'] == 'B':
        return row['market_cap']
    else:
        return row['market_cap']

# apply the function to the dataframe and create a new column 'market_cap_converted'
df['market_cap_converted'] = df.apply(convert_market_cap, axis=1)

In [149]:
# drop market cap	
df = df.drop(columns=['market_cap'])

# rename market cap converted to market cap
df = df.rename(columns={'market_cap_converted': 'market_cap'})

# columns order 'date', 'symbol', 'stock', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation', 'market cap um', 'market cap'
df = df[['date', 'symbol', 'stock', 'price', 'fair_value_(%)', 'z-score', 
'f-score', 'm-score', 'value_generation', 'market_cap_um', 'market_cap', 'count']]


In [150]:
# --- market cap filter ---
# drop all rows where market cap um is M
df = df[df['market_cap_um'] != 'M']

# drop all rows where market cap um is B and market cap is less than 10
df = df[~((df['market_cap_um'] == 'B') & (df['market_cap'] < 10))]

# rename df.market_cap to market_cap_(B)
df = df.rename(columns={'market_cap': 'market_cap_(B)'})

# drop market_cap_um
df = df.drop(columns=['market_cap_um'])

In [153]:
# add currency column according to the stock exchange where the stock is listed. Need to identify currency based on df.symbol 
# ends with:
# .MI -> EUR
# .DE -> EUR
# .PA -> EUR
# .L -> GBP
# .T -> JPY
# .AX -> AUD
# .SR -> SAR
# .HK -> HKD
# .TO -> CAD
# .NZ -> NZD
# .KL -> MYR
# .MX -> MXN
# if df.symbol does not contain ".", add USD as currency
# if none of the above conditions are met, currency 
def identify_currency(symbol):
    if symbol.endswith('.MI') or symbol.endswith('.DE') or symbol.endswith('.PA'):
        return 'EUR'
    elif symbol.endswith('.L'):
        return 'GBP'
    elif symbol.endswith('.T'):
        return 'JPY'
    elif symbol.endswith('.AX'):
        return 'AUD'
    elif symbol.endswith('.SR'):
        return 'SAR'
    elif symbol.endswith('.HK'):
        return 'HKD'
    elif symbol.endswith('.TO'):
        return 'CAD'
    elif symbol.endswith('.NZ'):
        return 'NZD'
    elif symbol.endswith('.KL'):
        return 'MYR'
    elif symbol.endswith('.MX'):
        return 'MXN'
    elif '.' not in symbol:
        return 'USD'
    else:
        return 'Unknown'
    
# apply the function to the dataframe and create a new column 'currency'
df['currency'] = df['symbol'].apply(identify_currency)

In [154]:
# print Unknown symbols in currency column
print(df[df['currency'] == 'Unknown']['symbol'])

Series([], Name: symbol, dtype: object)


In [155]:
# --- currency filter ---
# keep only EUR and USD stocks
df = df[df['currency'].isin(['EUR', 'USD'])]

In [156]:
# --- solidity filter ---
# convert z-score, f-score, m-score to numeric
df['z-score'] = pd.to_numeric(df['z-score'], errors='coerce')
df['f-score'] = pd.to_numeric(df['f-score'], errors='coerce')
df['m-score'] = pd.to_numeric(df['m-score'], errors='coerce')

# drop all rows where z-score is less than 1.81
# drop all rows where f-score is less than 3
# drop all rows where m-score is higher than -1.78
df = df[~((df['z-score'] < 1.81) | (df['f-score'] < 3) | (df['m-score'] > -1.78))]

In [162]:
# --- fair_value_(%) filter ---
# convert fair_value_(%) to numeric
df['fair_value_(%)'] = pd.to_numeric(df['fair_value_(%)'], errors='coerce')

# keep only rows where fair_value_(%) is higher than 0
df = df[df['fair_value_(%)'] > 0]

In [167]:
# list of tickers in the dataframe
tickers = df['symbol'].tolist()

# TODO: change it later, we will keep first 3 tickers for testing
tickers = tickers[:3]

tickers

['PYPL', 'CLX', 'ADBE']

In [166]:
# print length of dataframe
print("Number of rows in the dataframe:", len(df))

# print lenght of unique symbols in the dataframe (for comparison with the original dataframe)
print("Number of unique symbols in the dataframe:", df['symbol'].nunique())

Number of rows in the dataframe: 88
Number of unique symbols in the dataframe: 88


In [165]:
df.sample(10)

,date,symbol,stock,price,fair_value_(%),z-score,f-score,m-score,value_generation,market_cap_(B),count,currency
53,2026-02-01,UPS,"United Parcel Service, Inc.",110.50,19.74,3.28,8.0,-3.14,Robust,93.81,4,USD
9,2025-11-01,NOC,Northrop Grumman Corporation,566.70,3.36,3.07,7.0,-3.08,Robust,81.09,1,USD
116,2026-03-01,NKE,"NIKE, Inc.",51.41,24.61,6.86,5.0,-2.28,Robust,76.06,1,USD
12,2025-12-01,FFIV,"F5, Inc.",263.14,3.63,5.21,8.0,-4.46,Robust,15.17,2,USD
52,2026-02-01,LVS,Las Vegas Sands Corp.,56.07,51.86,2.51,7.0,-2.26,Robust,37.85,1,USD
29,2026-01-01,HLT,Hilton Worldwide Holdings Inc.,308.72,28.54,3.00,7.0,-2.32,Robust,73.17,3,USD
42,2026-02-01,AVY,Avery Dennison Corporation,187.20,29.80,3.93,9.0,-2.89,Robust,14.49,4,USD
7,2025-11-01,FOXA,Fox Corporation,65.69,6.93,3.15,6.0,-3.61,Robust,30.02,1,USD
92,2026-03-01,LEN,Lennar Corporation,86.05,39.20,3.99,4.0,-3.76,Robust,21.03,2,USD
65,2026-02-01,CTSH,Cognizant Technology Solutions Corporation,65.12,27.89,6.15,7.0,-2.41,Resilient,31.32,1,USD


In [159]:
# # Examples from finvizfinance documentation
# stock = finvizfinance('tsla')

# stock.TickerCharts(out_dir='asset')

# stock_fundament = stock.ticker_fundament()
# stock_description = stock.ticker_description()
# outer_ratings_df = stock.ticker_outer_ratings()
# news_df = stock.ticker_news()
# inside_trader_df = stock.ticker_inside_trader()